## Final work: Ising model and Monte Carlo method

The Ising model, proposed in 1925 by Ernst Ising, describes magnetic systems by means of spins that can only take the values +1 or -1. These spins are arranged in a lattice and tend to align with their neighbours, which minimises the energy of the system. Although it was devised to study ferromagnetism, its use has been extended to multiple disciplines. In this work, a two-dimensional lattice with periodic boundary conditions is analysed. The evolution of the system is studied using the Metropolis Monte Carlo algorithm. This method generates configurations according to their thermal probability, allowing phase transitions to be observed. Different temperatures are simulated to analyse the collective behaviour of the spins.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import colors


#===============================
# Ising Model Functions  
#===============================

# Size of the square lattice (50x50 spins)
N = 50  

def initialize_spins(N):
    """ Initializes an N x N spin lattice with all spins set to +1 """
    return np.ones((N, N), dtype=int)

def calculate_energy(spins):
    """
    Calculates the total energy of the spin configuration, which
    depends on the interaction between neighboring spins.
    """
    energy = 0
    for i in range(N):
        for j in range(N):
            # Sum of the 4 nearest neighbors (up, down, left, right)
            neighbors = spins[(i+1)%N, j] + spins[(i-1)%N, j] + spins[i, (j+1)%N] + spins[i, (j-1)%N]
            energy -= spins[i,j] * neighbors
    return energy / 2  # Divided by 2 to avoid double counting interactions

def metropolis_step(spins, T):
    """ 
    One step of the Metropolis algorithm for temperature T.
    Attempts to flip a random spin and accepts/rejects the change based on this magnitude.
    """
    i, j = np.random.randint(0, N, 2)  # Selects a random spin
    S = spins[i, j]

    # Calculation of interaction with neighbors
    neighbors = spins[(i+1)%N, j] + spins[(i-1)%N, j] + spins[i, (j+1)%N] + spins[i, (j-1)%N]
    delta_E = 2 * S * neighbors # Energy change if we flip the spin
    
    # Metropolis criterion: accept changes that decrease energy or with certain probability
    if delta_E <= 0 or np.random.random() < np.exp(-delta_E / T):
        spins[i, j] *= -1 # Flip the spin
        return delta_E  # Returns the energy change
    return 0 # No change occurred

def calculate_magnetization(spins):
    """
    Calculates the average magnetization (normalized between -1 and 1)
    Magnetization is the sum of all spins divided by the total number of spins.
    """
    return np.sum(spins) / spins.size


#===============================
# Ising Model Simulation
#===============================

def simulate_ising(N=50, temperatures=[1.0, 2.0, 2.269, 2.5, 3.0], mc_steps=100, eq_steps=100):
    """
    Simulates the Ising model for specified temperatures.
    
    Parameters:
    - N: Lattice size (N x N)
    - temperatures: List of temperatures to simulate
    - mc_steps: Monte Carlo steps for production
    - eq_steps: Equilibration steps

    Returns:
    - Final configurations, temperatures, magnetizations and energies
    """
    final_configs = []
    magnetizations = []
    energies = []
    
    for T in temperatures:
        print(f"Simulating T = {T:.3f}")
        spins = initialize_spins(N)
        energy = calculate_energy(spins)
        
        # Equilibration phase: let the system relax
        for _ in range(eq_steps * N**2):
            delta_E = metropolis_step(spins, T)
            energy += delta_E
        
        # Production phase: take measurements
        mag_sum = 0.0
        energy_sum = 0.0
        for _ in range(mc_steps):
            for _ in range(N**2):
                delta_E = metropolis_step(spins, T)
                energy += delta_E
            
            mag_sum += abs(calculate_magnetization(spins))  # Absolute magnetization
            energy_sum += energy / N**2  # Energy per spin
        
        # Save results for this temperature
        final_configs.append(spins.copy())
        magnetizations.append(mag_sum / mc_steps)
        energies.append(energy_sum / mc_steps)
    
    return final_configs, temperatures, magnetizations, energies

#===============================
# Results Visualization
#===============================

def print_results(temperatures, magnetizations, energies):
    """Displays numerical results in table format"""
    print("\nNumerical results:")
    print("\nTemperature | Magnetization | Energy per spin")
    print("----------------------------------------------")
    for T, mag, en in zip(temperatures, magnetizations, energies):
        print(f"{T:10.3f} | {mag:13.4f} | {en:15.4f}")


def plot_combined_results(temperatures, magnetizations, energies):
    """
    Plots two subplots:
    1. Magnetization vs temperature
    2. Energy vs temperature
    """
    plt.figure(figsize=(10, 8))
    
    # First subplot: Magnetization
    plt.subplot(2, 1, 1)  
    plt.plot(temperatures, magnetizations, 'o-', color='darkred', linewidth=2, label='Magnetization')
    plt.ylabel("Average magnetization (|m|)")
    plt.title("Magnetic and Energetic Behavior of the Ising Model")
    plt.grid(alpha=0.3)
    plt.legend()
    
    # Second subplot: Energy
    plt.subplot(2, 1, 2)  
    plt.plot(temperatures, energies, 's-', color='blue', linewidth=2, label='Energy')
    plt.xlabel("Temperature (T)")
    plt.ylabel("Average energy per spin (E/N²)")
    plt.grid(alpha=0.3)
    plt.legend()
    
    plt.tight_layout()
    plt.show()


def plot_spin_configurations(configs, temps):
    """ Plots the final spin configurations for each temperature"""
    fig, axes = plt.subplots(1, len(temps), figsize=(15, 3))
    cmap = colors.ListedColormap(['black', 'white'])  # -1: black, +1: white
    
    for ax, T, spins in zip(axes, temps, configs):
        ax.imshow(spins, cmap=cmap, vmin=-1, vmax=1)
        ax.set_title(f"T = {T:.3f}")
        ax.set_xticks([])
        ax.set_yticks([])
    
    plt.suptitle("Ising Model Spin Configurations")
    plt.tight_layout()
    plt.show()


#===============================
# Results Execution
#===============================

""" Define temperatures to simulate"""
our_temperatures = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]  

""" Run the simulation"""
configs, temps, mags, energies = simulate_ising(N=50, temperatures=our_temperatures, mc_steps=100, eq_steps=100)

""" Display results"""
print_results(temps, mags, energies)
plot_spin_configurations(configs, temps)
plot_combined_results(temps, mags, energies)

Simulating T = 0.000


C:\Users\Claudia Torres\AppData\Local\Temp\ipykernel_13636\2563454637.py:43: RuntimeWarning: divide by zero encountered in divide
  if delta_E <= 0 or np.random.random() < np.exp(-delta_E / T):


Simulating T = 1.000
Simulating T = 2.000
Simulating T = 3.000
Simulating T = 4.000
Simulating T = 5.000

Numerical results:

Temperature | Magnetization | Energy per spin
----------------------------------------------
     0.000 |        1.0000 |         -2.0000
     1.000 |        0.9993 |         -1.9973
     2.000 |        0.9084 |         -1.7398
     3.000 |        0.0526 |         -0.8199
     4.000 |        0.0354 |         -0.5636
     5.000 |        0.0266 |         -0.4268


In [ ]:
""""
IMPORTANT:

This real-time code was not part of the original project but we implemented it as an extra feature.

Normally, this code does not run correctly on the first try, so if it doesn't work the first time, try running it a second time.
"""

import matplotlib
matplotlib.use('TkAgg')
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Button
from matplotlib import colors
import time


#===========================================
#  Main Class, Initialization and Setup
#===========================================

class InteractiveIsing:
    def __init__(self, size = 50, initial_temperature=2.0):
        """
        Initializes the Ising model simulation with the following parameters:

        size - Size of the grid
        initial_temperature - Initial temperature of the system
        """

        self.size = size  # Grid size
        self.T = initial_temperature  # Initial temperature set to 2.0
        self.spins = np.ones((size, size), dtype=int)  # Initial configuration with all spins set to +1
        self.initial_state = False  # Initial state of the simulation, paused

        
        # Figure and subplot configuration
        self.fig, (self.ax_spins, self.ax_temp) = plt.subplots(1, 2, figsize=(10, 5))
        self.fig.subplots_adjust(bottom=0.3)  # Space for controls
        
        # Spin plot configuration
        self.color_map = colors.ListedColormap(['black', 'white'])  # -1 for black, +1 for white
        self.norm = colors.BoundaryNorm([-1, 0, 1], self.color_map.N)
        self.image = self.ax_spins.imshow(self.spins, cmap=self.color_map, norm=self.norm, 
                                interpolation='nearest')
        self.ax_spins.set_title(f'Temperature: {self.T:.2f}', pad=20)
        self.ax_spins.set_xticks([])
        self.ax_spins.set_yticks([])
        
        # Temperature control plot configuration
        self.ax_temp.set_xlim(0, 1)
        self.ax_temp.set_ylim(0, 5)
        self.temp_line, = self.ax_temp.plot([0.5, 0.5], [0, self.T], 'r-', lw=5)
        self.ax_temp.set_title('Temperature Control', pad=20)
        self.ax_temp.set_yticks(np.arange(0, 5.5, 0.5))
        self.ax_temp.set_xticks([])
        self.ax_temp.grid(True, alpha=0.3)
        
        # Creation of interactive buttons
        self.ax_increase = plt.axes([0.3, 0.05, 0.15, 0.075])    # Button to increase temperature
        self.ax_decrease = plt.axes([0.55, 0.05, 0.15, 0.075])   # Button to decrease temperature
        self.ax_toggle = plt.axes([0.4, 0.15, 0.2, 0.075])       # Start/pause button
        
        self.button_increase = Button(self.ax_increase, 'Increase T')
        self.button_decrease = Button(self.ax_decrease, 'Decrease T')
        self.button_toggle = Button(self.ax_toggle, 'Start/Pause')
        
        # Activate buttons
        self.button_increase.on_clicked(self.increase_temp)
        self.button_decrease.on_clicked(self.decrease_temp)
        self.button_toggle.on_clicked(self.toggle_simulation)
        
        # Timer configuration for animation with frame update every 50 ms
        self.timer = self.fig.canvas.new_timer(interval=50)  
        self.timer.add_callback(self.update)
        self.last_update = time.time()
    

#===============================
# Simulation Core Algorithm
#===============================

    def metropolis_step(self):

        #Performs one step of the Metropolis algorithm for the Ising model.
        
        """
        The algorithm:

        1. Randomly select a spin
        2. Calculate the energy change if the spin flips from -1 to +1 or vice versa
        3. Accept or reject the change following Boltzmann probability
        """
        
        # Randomly select a point on the grid
        i, j = np.random.randint(0, self.size, 2)
        
        # Periodic boundary conditions
        right = self.spins[i, (j+1)%self.size]
        left = self.spins[i, (j-1)%self.size]
        down = self.spins[(i+1)%self.size, j]
        up = self.spins[(i-1)%self.size, j]
        
        # Calculate energy change (ΔE)
        delta_E = 2 * self.spins[i,j] * (right + left + down + up)
        
        # Metropolis rule:
        # If -ΔE ≤ 0: always accept
        # If -ΔE > 0: accept with probability exp(-ΔE/T)
        probability = np.exp(-delta_E / self.T) if delta_E > 0 else 1.0
        
        if np.random.random() < probability:
            self.spins[i,j] *= -1


#==================================
# Simulation Update and Control
#==================================

    def update(self):
        
        #Updates the simulation and visualization.
        
        #Performs size squared Monte Carlo steps 

        if self.initial_state:
            # Perform a full Monte Carlo step
            for _ in range(self.size**2):
                self.metropolis_step()
            
            # Update visualization
            self.image.set_array(self.spins)
            self.ax_spins.set_title(f'Temperature: {self.T:.2f}', pad=20)
            self.temp_line.set_ydata([0, self.T])
            self.fig.canvas.draw_idle()
            self.last_update = time.time()
        
        # Restart timer if simulation is active
        if self.initial_state:
            self.timer.start()


#========================
# Temperature Control
#========================

    #Increases the temperature by 0.1 units up to a maximum of 5.0
    def increase_temp(self, event):
        self.T = min(5.0, self.T + 0.1)
        self.update_temp_display()

    #Decreases the temperature by 0.1 units down to a minimum of 0.1
    def decrease_temp(self, event):
        self.T = max(0.1, self.T - 0.1)
        self.update_temp_display()

    #Updates the temperature display
    def update_temp_display(self):
        self.temp_line.set_ydata([0, self.T])
        self.ax_spins.set_title('Temperature: self.T', pad=20)
        self.fig.canvas.draw_idle()
    

#=============================
# Simulation State Control
#=============================

    #Toggles simulation execution and pause
    def toggle_simulation(self, event):
        self.initial_state = not self.initial_state
        self.button_toggle.label.set_text('Pause' if self.initial_state else 'Continue')
        if self.initial_state:
            self.timer.start()
        else:
            self.timer.stop()
        self.fig.canvas.draw_idle()


#======================
# Results Execution
#======================

# Create and display the simulation
simulation = InteractiveIsing(size=50, initial_temperature=2.0)
plt.show()
